# A2 Q3 - Stage-1 baselines on Q3's own evaluation population (run locally)

Q3 reports NRMS against NRMS. This notebook puts A1's retrieval methods on
the **same impressions**, so the design note can say where a reproduced
neural ranker actually lands relative to BM25 and frozen-embedding cosine
rather than only relative to its own baseline arm.

It scores `bm25` and `embedding` over exactly the impressions staged in
`data/kaggle_nrms/nrms_{dataset}_{split}.parquet` - the same rows, in the
same order, that `nrms_baseline_kaggle.ipynb` scored - and writes:

| file | contents |
|---|---|
| `data/processed/{dataset}/stage1_per_impression_{split}.npz` | per-impression AUC/MRR/nDCG plus the impression ids, in staged-file order |
| `data/processed/{dataset}/stage1_vs_nrms.json` | point estimates with bootstrap CIs, the paired BM25-vs-embedding CI, and NRMS's numbers read back from `nrms_metrics_{dataset}.json` |

**Why this is cheap.** A2 Q2 §7 measured Stage-1 scoring at ~430
impressions/s on `ebnerd_large` and ~283/s on `mind_large`. The full
val+test populations would be ~9.2 hours on `ebnerd_large`; 200,000 per
split is ~16 minutes there and ~24 on `mind_large`. No GPU, no re-training,
no Kaggle.

**Why the per-impression arrays are persisted.** A paired CI between NRMS
and BM25 needs both methods' per-impression values indexed the same way.
The Kaggle notebook currently keeps only aggregates, so this file saves its
side of that pairing now; once a Kaggle run also emits its arrays, the
paired interval can be computed without re-scoring anything.

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.evaluation import (
    auc_impression,
    bootstrap_ci,
    mrr,
    ndcg_at_k,
    paired_bootstrap_ci,
)
from cs4406m26_assignment1c1.retrieval import RECENT_N_CLICKS, build_stage1_scorers


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
STAGED_DIR = ROOT / "data" / "kaggle_nrms"
PROGRESS_LOG = ROOT / "build_progress.log"

SPLITS = ["val", "test"]
METHODS = ["bm25", "embedding"]
NDCG_K_VALUES = [5, 10]
METRIC_NAMES = ["auc", "mrr"] + [f"ndcg{k}" for k in NDCG_K_VALUES]
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 0
LOG_EVERY = 25_000

# One dataset per kernel, same convention and same reason as
# RERANK_EVAL_DATASETS: a BM25 index plus an embedding matrix for both large
# tracks does not fit alongside each other on a 16GB machine.
_env = os.environ.get("STAGE1_DATASETS")
DATASETS = _env.split(",") if _env else ["ebnerd_large", "mind_large"]


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] stage1_baselines: {message}\n")
        f.flush()


def staged_path(dataset: str, split: str) -> Path:
    return STAGED_DIR / f"nrms_{dataset}_{split}.parquet"


def npz_path(dataset: str, split: str) -> Path:
    return DATA_DIR / dataset / f"stage1_per_impression_{split}.npz"


log_progress(f"started (datasets={DATASETS})")
{name: {split: staged_path(name, split).exists() for split in SPLITS} for name in DATASETS}

{'ebnerd_large': {'val': True, 'test': True}}

In [2]:
def test_setup() -> None:
    for name in DATASETS:
        for split in SPLITS:
            assert staged_path(name, split).exists(), f"{name}/{split}: run nrms_inputs.py first"
        for fname in ("articles.parquet", "history.parquet", "article_embeddings.parquet"):
            assert (DATA_DIR / name / fname).exists(), f"{name}: missing {fname}"
        # NRMS was scored with the same 20-click window; if these ever
        # diverge the comparison is between different inputs, not methods.
        assert RECENT_N_CLICKS == 20, RECENT_N_CLICKS
        nrms = DATA_DIR / name / f"nrms_metrics_{name}.json"
        if nrms.exists():
            meta = json.loads(nrms.read_text(encoding="utf-8"))
            assert meta["hyperparameters"]["history_size"] == RECENT_N_CLICKS
            for split in SPLITS:
                staged = pl.scan_parquet(staged_path(name, split)).select(pl.len()).collect().item()
                assert meta["population"][split]["impressions"] == staged, (name, split)
        else:
            print(f"   {name}: no nrms_metrics_{name}.json yet, NRMS columns will be omitted")


test_setup()
print("setup OK:", {name: [str(staged_path(name, s).name) for s in SPLITS] for name in DATASETS})

setup OK: {'ebnerd_large': ['nrms_ebnerd_large_val.parquet', 'nrms_ebnerd_large_test.parquet']}


## Score both Stage-1 methods over the staged impressions

Row order is preserved exactly as the staged parquet holds it, because that
is the order `nrms_baseline_kaggle.ipynb` built its own per-impression
arrays in (`load_split` reads the same file and never re-sorts). Position
*i* therefore means the same impression in both, which is the whole
premise of a paired comparison later. The impression ids are saved
alongside the metrics so that can be checked rather than trusted.

The staged file is already sorted by `user_id`, which is what makes the
BM25 scorer's one-entry cache hit: a user's impressions arrive
consecutively, so `get_scores` runs once per user rather than once per
impression.

In [3]:
def score_split(dataset: str, split: str, scorers: dict) -> dict[str, np.ndarray]:
    df = pl.read_parquet(
        staged_path(dataset, split),
        columns=["impression_id", "user_id", "article_ids_inview", "article_ids_clicked"],
    )
    n = df.height
    out = {
        f"{method}_{metric}": np.empty(n, dtype=np.float64)
        for method in METHODS
        for metric in METRIC_NAMES
    }
    impression_ids = df["impression_id"].to_list()

    started = datetime.now(timezone.utc)
    for i, (user_id, inview, clicked) in enumerate(
        zip(df["user_id"].to_list(), df["article_ids_inview"].to_list(), df["article_ids_clicked"].to_list())
    ):
        labels = np.fromiter((aid in set(clicked) for aid in inview), dtype=bool, count=len(inview))
        for method in METHODS:
            scored = scorers[method](user_id, inview)
            scores = np.fromiter((scored[aid] for aid in inview), dtype=np.float64, count=len(inview))
            out[f"{method}_auc"][i] = auc_impression(scores, labels)
            out[f"{method}_mrr"][i] = mrr(scores, labels)
            for k in NDCG_K_VALUES:
                out[f"{method}_ndcg{k}"][i] = ndcg_at_k(scores, labels, k)
        if (i + 1) % LOG_EVERY == 0:
            rate = (i + 1) / max((datetime.now(timezone.utc) - started).total_seconds(), 1e-9)
            log_progress(f"  {dataset}/{split}: {i + 1:,}/{n:,} impressions ({rate:.0f}/s)")

    # unicode dtype, not object: savez then needs no pickling on either side
    out["impression_id"] = np.asarray(impression_ids)
    return out


per_impression: dict[tuple[str, str], dict[str, np.ndarray]] = {}
for name in DATASETS:
    needed = [s for s in SPLITS if not npz_path(name, s).exists()]
    if needed:
        log_progress(f"  {name}: building Stage-1 scorers")
        scorers = build_stage1_scorers(
            articles=pl.read_parquet(
                DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract"]
            ),
            history_path=DATA_DIR / name / "history.parquet",
            embeddings_path=DATA_DIR / name / "article_embeddings.parquet",
        )
        for split in needed:
            result = score_split(name, split, scorers)
            # tmp must itself end in .npz or numpy appends the extension
            tmp = npz_path(name, split).with_suffix(".tmp.npz")
            np.savez_compressed(tmp, **result)
            os.replace(tmp, npz_path(name, split))
            log_progress(f"  {name}/{split}: written")
        # Released before the next dataset: the index and the embedding
        # matrix are the two large transients here (A2 Q2 §5).
        del scorers
    for split in SPLITS:
        # context manager, not a bare np.load: an open NpzFile keeps a
        # Windows file handle, and a later os.replace over it fails with
        # WinError 32 rather than anything diagnosable.
        with np.load(npz_path(name, split)) as loaded:
            per_impression[(name, split)] = {k: loaded[k] for k in loaded.files}

{f"{n}/{s}": len(per_impression[(n, s)]["bm25_auc"]) for n, s in per_impression}

{'ebnerd_large/val': 200000, 'ebnerd_large/test': 200000}

In [4]:
def test_scored() -> None:
    for name in DATASETS:
        for split in SPLITS:
            per = per_impression[(name, split)]
            staged = pl.read_parquet(staged_path(name, split), columns=["impression_id"])
            # Alignment, checked rather than assumed: a paired CI against
            # NRMS is only meaningful if position i is the same impression
            # in both, and both sides derive their order from this file.
            assert list(per["impression_id"]) == staged["impression_id"].to_list(), (
                f"{name}/{split}: per-impression order does not match the staged file"
            )
            for method in METHODS:
                for metric in METRIC_NAMES:
                    values = per[f"{method}_{metric}"]
                    assert len(values) == staged.height, (name, split, method, metric)
                    assert np.isfinite(values).all(), f"{name}/{split}/{method}/{metric}: non-finite"
                    assert values.min() >= 0.0 and values.max() <= 1.0, (values.min(), values.max())
            # The two methods must not produce identical rankings; if they
            # did, one adapter is silently returning the other's scores.
            assert not np.array_equal(per["bm25_auc"], per["embedding_auc"]), (name, split)
            print(
                f"   {name}/{split}: "
                + "  ".join(f"{m}={per[f'{m}_auc'].mean():.4f}" for m in METHODS)
            )


test_scored()
print("scored population OK (aligned to the staged file, bounded, methods distinct)")

   ebnerd_large/val: bm25=0.5088  embedding=0.5597
   ebnerd_large/test: bm25=0.5015  embedding=0.5511
scored population OK (aligned to the staged file, bounded, methods distinct)


## Confidence intervals and the comparison against NRMS

Two kinds of interval, and the difference matters:

- **Bootstrap CI** per method, on its own metric. Reported for every
  method here and already present for every NRMS variant in
  `nrms_metrics_{dataset}.json`.
- **Paired bootstrap CI** on a difference, which needs both methods'
  per-impression arrays. BM25 vs embedding is computed here, since both
  sides exist. NRMS vs Stage-1 is *not* computed: the Kaggle notebook
  persisted aggregates only, so the pairing has no NRMS array to pair
  against yet. The gap is recorded in the output rather than papered over
  with an unpaired comparison of two independent intervals, which would be
  the weaker test (A2 Q2 §8).

In [5]:
def build_payload(dataset: str) -> dict:
    nrms_path = DATA_DIR / dataset / f"nrms_metrics_{dataset}.json"
    nrms = json.loads(nrms_path.read_text(encoding="utf-8")) if nrms_path.exists() else None

    stage1 = {}
    for method in METHODS:
        stage1[method] = {}
        for split in SPLITS:
            per = per_impression[(dataset, split)]
            stage1[method][split] = {
                metric: dict(
                    zip(
                        ("point", "ci_lo", "ci_hi"),
                        bootstrap_ci(per[f"{method}_{metric}"], BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED),
                    )
                )
                for metric in METRIC_NAMES
            }

    paired = {}
    for split in SPLITS:
        per = per_impression[(dataset, split)]
        entry = {}
        for metric in METRIC_NAMES:
            diff, lo, hi = paired_bootstrap_ci(
                per[f"bm25_{metric}"], per[f"embedding_{metric}"], BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED
            )
            entry[metric] = {
                "mean_diff": diff,
                "ci_lo": lo,
                "ci_hi": hi,
                "excludes_zero": bool(lo > 0 or hi < 0),
            }
        paired[split] = entry

    payload = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset": dataset,
        "population": {
            split: {"impressions": len(per_impression[(dataset, split)]["bm25_auc"])} for split in SPLITS
        },
        "hyperparameters": {
            "recent_n_clicks": RECENT_N_CLICKS,
            "ndcg_k_values": NDCG_K_VALUES,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": BOOTSTRAP_SEED,
        },
        "stage1_metrics": stage1,
        "paired_bm25_vs_embedding": paired,
        "nrms_metrics": (
            {variant: nrms["ranking_metrics"][variant] for variant in nrms["ranking_metrics"]}
            if nrms
            else None
        ),
        "paired_nrms_vs_stage1": None,
        "notes": (
            "NRMS point estimates are read from nrms_metrics_{dataset}.json, measured on the same "
            "staged impressions. A paired CI against them is not computed here because that run "
            "persisted aggregates only, not per-impression arrays."
        ),
    }
    path = DATA_DIR / dataset / "stage1_vs_nrms.json"
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(tmp, path)
    return payload


payloads = {name: build_payload(name) for name in DATASETS}
log_progress("stage1_vs_nrms.json written for " + ", ".join(DATASETS))
sorted(str(DATA_DIR / n / "stage1_vs_nrms.json") for n in DATASETS)

['C:\\Users\\HP\\Desktop\\Coursework\\IRE\\cs4406m26-assignment1c1\\data\\processed\\ebnerd_large\\stage1_vs_nrms.json']

In [6]:
def test_payload() -> None:
    for name in DATASETS:
        payload = json.loads((DATA_DIR / name / "stage1_vs_nrms.json").read_text(encoding="utf-8"))
        for method in METHODS:
            for split in SPLITS:
                for metric in METRIC_NAMES:
                    e = payload["stage1_metrics"][method][split][metric]
                    assert 0.0 <= e["ci_lo"] <= e["ci_hi"] <= 1.0, (name, method, split, metric, e)
                    assert e["ci_lo"] - 1e-6 <= e["point"] <= e["ci_hi"] + 1e-6, e
        for split in SPLITS:
            for metric in METRIC_NAMES:
                e = payload["paired_bm25_vs_embedding"][split][metric]
                assert e["ci_lo"] - 1e-6 <= e["mean_diff"] <= e["ci_hi"] + 1e-6, e
                assert e["excludes_zero"] == (e["ci_lo"] > 0 or e["ci_hi"] < 0)
        # A1 already measured these two methods over the *full* val/test
        # populations. The sampled estimate should land near them; a large
        # gap would mean the staged sample is not representative.
        full = DATA_DIR / name / "eval_metrics.json"
        if full.exists():
            reference = json.loads(full.read_text(encoding="utf-8"))
            for method in METHODS:
                for split in SPLITS:
                    try:
                        ref = reference["ranking_metrics"][method][split]["auc"]["point"]
                    except (KeyError, TypeError):
                        continue
                    got = payload["stage1_metrics"][method][split]["auc"]["point"]
                    print(f"   {name}/{split}/{method}: sampled {got:.4f} vs full population {ref:.4f}")
                    assert abs(got - ref) < 0.01, (name, method, split, got, ref)
        else:
            print(f"   {name}: no eval_metrics.json in this checkout, full-population check skipped")


test_payload()
print("payload OK (CIs ordered, flags consistent)")

   ebnerd_large: no eval_metrics.json in this checkout, full-population check skipped
payload OK (CIs ordered, flags consistent)


## The table the design note needs

In [7]:
for name in DATASETS:
    payload = payloads[name]
    print(f"\n### {name} ".ljust(78, "#"))
    header = "method".ljust(20) + "split".ljust(7) + "".join(m.ljust(26) for m in METRIC_NAMES)
    print(header)
    rows = [(m, payload["stage1_metrics"][m]) for m in METHODS]
    if payload["nrms_metrics"]:
        rows += [(f"nrms/{v}", payload["nrms_metrics"][v]) for v in payload["nrms_metrics"]]
    for label, block in rows:
        for split in SPLITS:
            cells = []
            for metric in METRIC_NAMES:
                e = block[split][metric]
                cells.append(f"{e['point']:.4f} [{e['ci_lo']:.4f},{e['ci_hi']:.4f}]".ljust(26))
            print(label.ljust(20) + split.ljust(7) + "".join(cells))
    print("\npaired 95% CI, bm25 -> embedding (same impressions):")
    for split in SPLITS:
        e = payload["paired_bm25_vs_embedding"][split]["auc"]
        flag = "excludes zero" if e["excludes_zero"] else "INCLUDES ZERO"
        print(f"   {split:<5} d_auc={e['mean_diff']:+.4f} [{e['ci_lo']:+.4f}, {e['ci_hi']:+.4f}]  {flag}")


### ebnerd_large ############################################################
method              split  auc                       mrr                       ndcg5                     ndcg10                    
bm25                val    0.5088 [0.5074,0.5103]    0.3361 [0.3348,0.3374]    0.3708 [0.3694,0.3723]    0.4541 [0.4529,0.4554]    
bm25                test   0.5015 [0.5001,0.5029]    0.3191 [0.3178,0.3204]    0.3495 [0.3480,0.3510]    0.4344 [0.4331,0.4356]    
embedding           val    0.5597 [0.5583,0.5610]    0.3639 [0.3626,0.3652]    0.4075 [0.4061,0.4089]    0.4829 [0.4818,0.4840]    
embedding           test   0.5511 [0.5496,0.5525]    0.3486 [0.3472,0.3498]    0.3867 [0.3852,0.3882]    0.4648 [0.4636,0.4660]    
nrms/baseline       val    0.5701 [0.5687,0.5714]    0.3678 [0.3665,0.3690]    0.4109 [0.4094,0.4124]    0.4895 [0.4883,0.4905]    
nrms/baseline       test   0.5835 [0.5821,0.5848]    0.3714 [0.3700,0.3727]    0.4122 [0.4106,0.4136]    0.4859 [0.4846,0.4870]  

In [8]:
def test_table_inputs() -> None:
    # Every number printed above must come from the persisted file, not from
    # whatever happens to be in memory at the end of a partial run.
    for name in DATASETS:
        on_disk = json.loads((DATA_DIR / name / "stage1_vs_nrms.json").read_text(encoding="utf-8"))
        assert on_disk["stage1_metrics"] == payloads[name]["stage1_metrics"], name
        for split in SPLITS:
            assert on_disk["population"][split]["impressions"] == len(
                per_impression[(name, split)]["bm25_auc"]
            )


test_table_inputs()
log_progress("finished")
print("table OK (printed values match the persisted file)")

table OK (printed values match the persisted file)


# Manual Verification Complete

`data/processed/{dataset}/stage1_vs_nrms.json` now carries BM25 and
embedding measured on Q3's exact evaluation population, with NRMS's own
numbers alongside for the design note's table.

One gap remains deliberate: the NRMS-vs-Stage-1 comparison is
point-estimate only. To upgrade it to a paired interval, a Kaggle run has
to persist its per-impression metric arrays in the same staged-file order
this notebook uses - `nrms_protocol_fix_kaggle.ipynb` does that, so once it
has run, the pairing can be computed here without re-scoring anything.